# 第4回 演習：前処理（欠測・標準化・カテゴリ）

## 前回からの接続

第3回では、データを訓練とテストに分けて「新しい日での実力」を測る足場を組み、最後にデータリークという落とし穴まで見ました。ただしそこで扱ったデータは、値の抜けもなく単位もそろった、あらかじめ整えられたものでした。

今日はその一つ手前の工程に降ります。穴の空いた列を埋め、桁のちがう列の土俵をそろえ、数値の顔をしたカテゴリを 0/1 に開く——現場から届いたままのデータを、解析できる形に直す作業です。そして第3回が予告したとおり、前処理こそ「先に分割してから」の順序を破るとリークが起きる、いちばん危ない場所でもあります。


## 今日の分析目標

**欠損・単位ちがい・カテゴリ列を、解析できる形に整えたい。**

この演習では、欠測の補完・標準化・カテゴリのワンホット化という「三点セット」と、それらを訓練データだけで学ぶ正しい順序を、自転車データで自分の手で動かします。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

## 学習ゴール

この回を終えると、次のことができるようになります。

- 欠測のある行を削除する代償を日数で示し、なぜ削除より補完を選ぶのかを説明できる
- 欠測の理由を MCAR・MAR・MNAR に区別し、中央値補完がどこまで正当化されるかを言える
- 標準化が分布の形を変えないこと、そして効くのは距離や正則化を使う手法で、木系には要らないことを説明できる
- 数値の顔をしたカテゴリをワンホットに開き、係数を「基準カテゴリからの差」として読める
- 補完・標準化・ワンホットの基準を訓練データだけで学ぶ順序を、リークの経路とセットで説明できる


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {2}
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. データの読み込みと欠測の確認

自転車データを読み込み、気温・湿度・風速の3列にわざと1割の欠測（NaN）を入れます。まず「どれくらい欠けているか」「削除するとどうなるか」を確認します。

### 深掘り：なぜ「削除」ではなく「補完」なのか、そしてなぜ中央値なのか

穴の空いたデータを前にして、いちばん簡単なのは**欠測のある行をまるごと捨てる**ことです。でもこれには大きな代償があります。この節のコードセルで確かめるとおり、このデータは 731 日ぶん。気温・湿度・風速の**3 列に 1 割（各 73 件）**の穴を空けただけなのに、`dropna()` で行削除すると残るのは **534 日**、**197 日ぶん**（全体の約 27％）が消えます。穴は 3 列にしかないのに、行を単位で捨てると、その日の**無傷だった残り列の値まで道連れ**になるからです。列が増えるほど「どこか 1 列は欠けている」行が積み上がり、削除は雪だるま式に効いてきます。だから実務では、削除より**補完**——穴をそれらしい値で埋めて活かす——が基本になります。

ここで数字を一つ突き合わせておくと理解が深まります。空けた穴（NaN のマス目）は $73 \times 3 = 219$ 個。ところが削除される日は **197 日**で、219 より少ない。差の 22 は、**同じ日に 2 つ以上の列が同時に欠けた**ぶんです（その日は 1 回数えれば足りる）。もし 3 列の欠測が完全に別々の日に散っていれば 219 日ぶん消えていたはずで、実際はたまたま重なった日があるおかげで 197 日で済んでいる、という理屈です。いずれにせよ、たった 3 列・1 割の欠測でここまで失うのは、削除の代償の大きさを物語っています。

**なぜ平均ではなく中央値か**　補完の基本は、その列の**代表値**で穴を埋めることです。左右対称なきれいな山なら、平均でも中央値でも大きくは変わりません。分かれるのは、片側に裾を引いた分布のとき。平均は少数の極端な値にぐいっと引っぱられますが、**中央値は「順位の真ん中」なので極端値に鈍感**です。このデータの `windspeed`（風速）は下限が 0 で、たまに強風の日が右へ裾を伸ばしがちな列——こういう列こそ、数個の強風日に代表値を吊り上げられない中央値が安全です。迷ったら中央値、が無難な第一手。TODO① で `strategy='median'` を選ぶのはこのためです。

**「どんな理由で欠けたか」も本当は効く**　欠測には、まったくの偶然で欠ける場合（MCAR）と、他の値や欠測値そのものに関係して欠ける場合（MAR・MNAR）があります。三者の違いを自転車データで想像してみましょう。センサーがランダムに故障して気温が抜けるなら **MCAR**（完全にランダム）——単純な中央値補完で困りません。「寒い日ほど計測を省きがち」で欠測が**他の観測できる状況（季節など）に依存**するなら **MAR**、「強すぎて測れなかった風速だけが抜ける」ように**欠測値そのものの大きさに依存**するなら **MNAR** です。MAR・MNAR では、観測できている値だけの中央値で埋めると、抜けやすかった側（寒い日・強風）を過小評価して分布をゆがめかねません。この演習では穴を人工的に**ランダムに**空けている（＝MCAR とみなせる）ので中央値補完で十分ですが、現実では「なぜ欠けたか」を一度疑う癖が要ります。三つの区別と、それが補完の正しさをどう左右するかは、次の深掘りで例を挙げて見ていきます。

**伏線**　ここで使う「中央値」も、次節の標準化の平均・ばらつきも、後で出るカテゴリの顔ぶれも、すべて**データから計算して求めた値＝学習したパラメータ**です。だとすればこれらは、モデルと同じく**訓練データだけから学ぶ**べきもの——この視点が、この回の背骨であるリーク防止（4 節）へまっすぐつながります。

### 深掘り：なぜ欠けたのか——MCAR・MAR・MNAR と多重代入

補完してよいかどうかは、じつは「どんな値で埋めるか」より前に、**なぜ欠けたか**で決まります。MCAR・MAR・MNAR という三つの区別を、身近な例で一つずつ見ておきましょう。

**MCAR（完全にランダムな欠測）**　欠けるかどうかが、観測されている値とも、欠けた値そのものとも無関係な場合です。たとえばアンケートの束を運ぶ途中で何枚かが雨に濡れて読めなくなった——どの回答が濡れるかは、回答の中身とも回答者の属性とも関係ありません。この演習で空けている穴も、1節のコードで `rng.choice` を使った無作為抽出なので、まさにこれ。MCAR なら、**残っている値は本来の分布からの公平な抜き取り**なので、その中央値は欠けた値まで含めた中央値の素直な推定になります。だから中央値で埋めても分布の中心はずれません（ただし全員を同じ値にするぶん、ばらつきは実際より小さくなります）。

**MAR（他の観測値で説明できる欠測）**　欠けやすさが、**手元に残っている別の列**で説明できる場合です。たとえば健康診断で、若い人ほど問診票の血圧欄を書き飛ばす。欠けやすさは年齢という観測できる列で決まっていて、**同じ年齢層の中だけで見れば**欠測は無作為です。このとき列全体の中央値で一律に埋めると、若い人が多く欠けたぶん分布が高齢側へ寄ってしまいます。正しくは年齢層ごとの中央値で埋める、つまり**条件つきで補完する**。中央値補完が救えるのはここまで、と覚えてください。

**MNAR（欠けた値そのものに依存する欠測）**　欠けやすさが、**見えていないその値の大きさ**で決まる場合です。収入を尋ねると収入の高い人ほど答えない、風速計が強風で振り切れて記録が残らない——観測できているのは低いほうに偏った値ばかりなので、その中央値で埋めれば全体を系統的に過小評価します。補完がバイアスを持ち込む、といういちばん厄介な状況です。しかも手元のデータだけからは MNAR かどうかを確かめる術がありません（欠けた値が見えないのだから当然です）。ここは統計の技術ではなく、**なぜ欠けたかについての現場の知識**で補うしかない領域です。

**多重代入という発想**　もう一つ、中央値補完には MCAR でも残る弱点があります。埋めた値を観測した値と同じ確かさで扱ってしまうため、後段の分析が誤差やばらつきを実際より小さく見積もるのです。**多重代入**は、1 つの値で埋めきらないことでここを直します。もっともらしい値を**ばらつきを持たせて複数回引き当て**、埋め方の違う複数のデータセットを作り、それぞれで同じ分析を回して、最後に結果を統合する。「埋めた値には不確かさがある」ことを、答えの幅として最後まで持ち越す考え方です。この演習では扱いませんが、発想だけ覚えておいてください。どんな仮定（とくに MAR）のもとで複数の結果を統合してよいのか、その理論的な裏づけは、多重代入を最初に定式化した Rubin の原論文と、欠測データ解析の専門書に譲ります。


In [ ]:
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
rng = np.random.default_rng(42)
for col in ['temp', 'hum', 'windspeed']:
    idx = rng.choice(df.index, size=int(len(df) * 0.1), replace=False)
    df.loc[idx, col] = np.nan

print('列ごとの欠測数:')
print(df[['temp', 'hum', 'windspeed', 'season']].isnull().sum())
print(f'\n元の日数: {len(df)} / dropna後: {len(df.dropna())}')

### TODO①：欠測を中央値で補完する

`SimpleImputer` を使って、数値列 `['temp','atemp','hum','windspeed']` の欠測を**中央値**で補完し、補完後の NaN の数が 0 になったことを確認してください。

外れ値に強いのは、平均と中央値のどちらだったかを思い出しましょう。

In [ ]:
num_cols = ['temp', 'atemp', 'hum', 'windspeed']
# TODO: SimpleImputer で df[num_cols] の欠測を中央値で補完し、補完前後の NaN 数を表示してください
# ヒント: 埋め方は strategy 引数で指定します。基準を学んで適用する、この一連の前処理で何度も出てくるメソッドを使います
...

## 2. 標準化でスケールをそろえる

TODO①の答え合わせも兼ねて、中央値での補完から標準化までを通します。標準化の前後で、平均とばらつきがどう変わるかを確認しましょう。

### 深掘り：標準化とは何か——zスコアの式・形を変えない理由・木系に要らない理由・訓練で学ぶ順序

**式の骨子**　標準化は、各列を次の値に置きかえる操作です。

$$
z \;=\; \frac{x - \mu}{\sigma}
$$

$x$ はもとの値、$\mu$（ミュー）はその列の**平均**、$\sigma$（シグマ）はその列の**標準偏差**（ばらつき）。平均を引いて中心を 0 に寄せ、標準偏差で割って幅を 1 にそろえます。つまり $z$ は「**その値が、平均から標準偏差何個ぶん離れているか**」という、単位のない物差しに乗せ替えたもの。摂氏でも人数でも、みな同じ「ばらつき何個ぶん」の言葉に翻訳されます。

**この演習の実データで**　標準化の前、4 列は中心も幅もバラバラです（平均は temp 0.495・hum 0.628・windspeed 0.188、ばらつきは temp 0.175・hum 0.135・windspeed 0.073 …）。標準化の後は、TODO② で確かめるとおり**どの列も平均がほぼ 0（実際は $10^{-16}$ オーダーのほぼゼロ）・標準偏差が 1.000** にそろいます。全部の変数が同じものさしに乗った状態です。

**なぜ「形」は変わらないのか**　$z=(x-\mu)/\sigma$ は $x$ の**一次式（アフィン変換）**——平行移動（$-\mu$）と拡大縮小（$\div\sigma$）だけです。値の順位も、山が 1 つか 2 つかも、裾の長さも保たれます。だからヒストグラムの**形はそっくりのまま、横軸の目盛りだけが変わる**。標準化は分布をゆがめず、情報を 1 つも捨てません。安心して使える素直な変換です。

**なぜ距離・正則化には要り、木系には要らないのか**　距離で予測する手法（k 近傍法など）や、係数の大きさに罰金をかける正則化は、変数の**大きさをそのまま**受け取ります。すると幅の広い列だけが幅をきかせ、幅の狭い列の効きがかすむ——だから土俵をそろえる標準化が要ります。いっぽう決定木やランダムフォレストのような**木系のモデル**は、各変数を「あるしきい値で上と下に分ける」形だけで使います。しきい値による大小比較は、列を定数で割って幅を変えても**順序が変わらない**ので、分割は 1 ミリも動きません。だから木系に標準化は不要です（この違いは、後の回で木系モデルを扱うときに効いてきます）。

**つまずきどころ：訓練で学び、テストに適用する順序（リーク防止）**　標準化の $\mu,\sigma$ も、前節で述べたとおり**データから学んだ値**です。だから正しくは、**訓練データだけで $\mu,\sigma$ を学び（`fit`）、テストにはその基準を当てはめるだけ（`transform`）**。全データをまとめて `fit` すると、テストの情報が $\mu,\sigma$ ににじみ込む——これが**リーク**です。

ここで正直な但し書きを一つ。この演習の主役である**線形回帰（最小二乗）は、じつは特徴量のアフィンな拡大縮小に対して予測が不変**です。仕組みは単純で、たとえば `temp` を標準化すると（$\div 0.175$ 相当で）列の値が約 5.7 倍に引き伸ばされますが、最小二乗はその列の**係数を逆に約 $0.175$ 倍に縮める**だけ。積 $\beta_{\text{temp}}\times(\text{temp}$ の列$)$ は変わらないので、予測 $X\beta$ も、当てはまりも一切動きません。列ごとに引く定数（平均）は切片が丸ごと吸収します。実際、4 節で使う「整えたデータ」で試すと、**標準化なし・訓練だけで学ぶ（正しい）・全データで学ぶ（リーク）の 3 通りとも、線形回帰のテスト R² は 0.8438 で小数点 4 桁まで完全に一致**します。つまり線形回帰では、標準化のリークは結果にまったく出ません。標準化は線形回帰の**精度を上げる操作ではなく、係数を無単位にして公平に見比べられるようにする**操作なのだ、とも言えます。

では油断してよいかというと、**ダメ**です。同じリークでも、距離や正則化を使うモデルでは $\mu,\sigma$ の違いが予測に効くので、全データで `fit` したリークはテスト性能を（見かけ上）動かします。要するに手法によって「バレるかバレないか」が違うだけで、**正しい順序は手法に依存せず `fit`＝訓練・`transform`＝テストで統一**しておくのが安全なのです。なぜ線形回帰だけがこのアフィン不変性を持つのか——予測は説明変数の列が張る空間へ目的変数を直交射影したもので、列を定数倍しても平行移動しても張る空間そのものは変わらないから——を、射影行列と正規方程式で厳密に書き下す議論は、回帰を行列と射影の言葉で書き直した専門書の付録に譲ります。

In [ ]:
# 図：fit / transform の順序（左）と、テストの情報が漏れる経路（右）
from matplotlib.patches import FancyBboxPatch

def _box(ax, x, y, w, h, text, fc, fs=11):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.012',
                                facecolor=fc, edgecolor='none'))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', color='white', fontsize=fs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, leak in zip(axes, [False, True]):
    ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title('リーク：テストも込みで fit してしまう' if leak
                 else '正しい順序：fit は訓練だけ', fontsize=13)
    _box(ax, 0.10, 0.85, 0.80, 0.12, '手元の全データ', '#888888')
    ax.annotate('', xy=(0.30, 0.72), xytext=(0.40, 0.845),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.6))
    ax.annotate('', xy=(0.80, 0.72), xytext=(0.62, 0.845),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.6))
    _box(ax, 0.06, 0.58, 0.46, 0.14, '訓練データ', '#0066cc')
    _box(ax, 0.62, 0.58, 0.32, 0.14, 'テスト', '#e76f51')
    _box(ax, 0.14, 0.12, 0.72, 0.16,
         '前処理の基準\n中央値・平均 $\\mu$・ばらつき $\\sigma$・カテゴリの顔ぶれ', '#2a9d8f')
    ax.annotate('', xy=(0.26, 0.29), xytext=(0.26, 0.57),          # fit：訓練 → 基準
                arrowprops=dict(arrowstyle='->', color='#0066cc', lw=2.4))
    ax.text(0.245, 0.43, 'fit', ha='right', va='center', color='#0066cc', fontsize=12)
    c_tr = '#e63946' if leak else '#555'                            # transform：基準 → 訓練
    ax.annotate('', xy=(0.42, 0.57), xytext=(0.42, 0.29),
                arrowprops=dict(arrowstyle='->', color=c_tr, lw=1.4))
    ax.text(0.435, 0.43, 'transform', ha='left', va='center', color=c_tr, fontsize=11)
    ax.annotate('', xy=(0.78, 0.57), xytext=(0.78, 0.29),           # transform：基準 → テスト
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.4))
    ax.text(0.795, 0.43, 'transform', ha='left', va='center', color='#555', fontsize=11)
    if leak:
        ax.annotate('', xy=(0.68, 0.29), xytext=(0.68, 0.57),       # fit：テスト → 基準（漏れ）
                    arrowprops=dict(arrowstyle='->', color='#e63946', lw=2.4))
        ax.text(0.665, 0.43, 'fit', ha='right', va='center', color='#e63946', fontsize=12)
        ax.text(0.5, 0.035, 'テストの値が基準に混ざり、訓練側へ戻る＝リーク',
                ha='center', va='center', color='#e63946', fontsize=12)
    else:
        ax.text(0.5, 0.035, 'テストは transform を受け取るだけ',
                ha='center', va='center', color='#0066cc', fontsize=12)
plt.tight_layout(); plt.show()


**この図の読み方**　左が正しい順序、右がリークです。どちらも上の灰色が手元の全データ、青が訓練データ、オレンジがテスト、下の緑が前処理の基準——補完の中央値、標準化の $\mu$ と $\sigma$、カテゴリの顔ぶれ——を入れる箱です。下向きの太い矢印が `fit`（基準を学ぶ向き）、上向きの細い矢印が `transform`（学んだ基準を当てはめる向き）。左では緑の箱へ入る矢印が**訓練からの青い1本だけ**で、テストは transform を受け取るだけ。だからテストの値は基準に一切入りません。

右で増えているのは**赤い矢印2本**です。1本目はテストから緑の箱へ下りる `fit`——分割の前に全データへ `fit_transform` を呼ぶのは、この矢印を足すのと同じことです。2本目はそこから訓練へ戻る `transform` で、これが**漏れの経路**。テストの値が混ざった基準が訓練側の変換に使われ、モデルは前処理を通じてテストを間接的に覗いています。左右の図は形がほとんど同じで、違いは赤い矢印の有無だけ——コードの上でも、`fit` を分割の前に呼ぶか後に呼ぶかという一行の差でしかありません。だから目で見て気づけず、順序を機械的に守るしかないのです。


In [ ]:
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(df[num_cols])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print('補完＋標準化 完了')

### TODO②：標準化の前後で平均とばらつきを確かめる

`X_imputed`（標準化前）と `X_scaled`（標準化後）のそれぞれについて、各列の**平均**と**標準偏差**を計算して比べてください。標準化後は平均がほぼ0・標準偏差がほぼ1になっているでしょうか。

In [ ]:
# TODO: 標準化前（X_imputed）と標準化後（X_scaled）の各列について、平均と標準偏差を表示してください
# ヒント: 「列ごと」に計算するには axis の指定が必要です
...

## 3. カテゴリ変数をワンホットにする

`season`（1〜4）や `weathersit`（1〜3）は、数値の顔をしたカテゴリでした。数字の大小に意味はありません。**ワンホットエンコーディング**で 0/1 の列に展開します。

### 深掘り：なぜ数値化が要るのか（第1・2回の伏線回収）／ダミー変数の罠と基準カテゴリ／順序 vs 名義／ターゲットエンコーディング

**第1・2回の宿題を回収する**　`season`（1〜4）や `weathersit`（1〜3）は、第1回の演習でデータを眺めたときに「**数値の顔をしたカテゴリ**」として見つけた列でした。数字で入ってはいますが、大小や間隔に意味はありません（季節 4 は季節 1 の 4 倍ではない）。ところが第2回の回帰では、これを**生の数字のまま**説明変数に入れていました。すると回帰は「`season` が 1 増えると台数が $\beta$ だけ増える」という**等間隔で一直線な効き**を勝手に仮定します——ありもしない順序と等間隔をモデルに教え込んでしまう。ずっと保留してきたこの宿題に、ここで正しく手を付けます。

なぜ「一直線」がまずいか、実データで一目です。季節ごとの平均利用台数を並べると、区分1 **2604** → 区分2 **4992** → 区分3 **5644** → 区分4 **4728** と、**いったん上がってまた下がる山型**で、番号の順にまっすぐ増えてはいません（そもそも `season` の番号は寒暖の順ですらなく、平均気温が最も高いのは区分3 です）。生の数値で入れると、回帰はこの山型を無理やり 1 本の直線で近似しようとして、山のてっぺんも下り坂も取りこぼします。番号の大小に意味がないカテゴリを、大小のある数として食べさせてしまう危うさが、この山型に表れています。

**なぜ数値化が必要で、one-hot がその答えか**　モデルは数しか食べられないので、カテゴリは何らかの形で数値化せねばなりません。でも番号をそのまま入れれば上の架空順序が混じる。そこで**ワンホットエンコーディング**——値ごとに 0/1 の列を作り、該当する区分の列だけ 1、ほかは 0 にします。順序を持ち込まず、各区分を独立に扱えます。実データでは `season` と `weathersit` を展開すると `season_1〜season_4`・`weathersit_1〜weathersit_3` の**7 列**ができます。

**ダミー変数の罠と基準カテゴリ**　ここに落とし穴があります。切片（定数項）を持つモデルに、あるカテゴリの**全**ダミー列を入れると、それらの合計が常に 1（`season_1+season_2+season_3+season_4=1`）となり、**切片の列とぴったり重なって**しまいます。列同士に完全な一次従属ができ、係数が一意に決まらない——これが**ダミー変数の罠**（多重共線性）です。回避は簡単で、各カテゴリから**1 列だけ落とす**（`drop_first=True`）。落とした水準が**基準カテゴリ（baseline）**になり、残った列の係数は「基準からの差」を表します。実データでは `drop_first` すると `season_2, season_3, season_4, weathersit_2, weathersit_3` の**5 列**になり、基準は `season_1`（季節の区分1）と `weathersit_1`（天気の区分1）。たとえば `season_4` の係数は「基準の区分1に対して、区分4 の日は利用がどれだけ多い／少ないか」を意味します。

正直に補足すると、`sklearn` の線形回帰は擬似逆行列で解くので、フル（全列）でも `drop_first` でも**テスト R² は同じ 0.844**——予測としては壊れません（設計行列で見ると、フルは切片込み 34 列でもランクは 29 で、罠のぶんだけ縮退しています）。それでも `drop_first` を勧めるのは、**係数を一意に、しかも「基準からの差」として読めるようにする**ためです。精度ではなく解釈のための作法だ、という点は正則化の回の教訓とも通じます。

**発展：順序尺度 vs 名義尺度**　カテゴリにも二種類あります。`season`（春夏秋冬）は本質的な順序の薄い**名義尺度**で、one-hot が素直な選択。いっぽう「低・中・高」のように**本来の順序**を持つもの（順序尺度）は、`0,1,2` と順序を保った整数に写す（ラベルエンコーディング）ほうが、順序という情報を活かせることもあります。名義を無理に整数化しない／順序を無理に捨てない——この使い分けが勘所です。

**発展：ターゲットエンコーディングという一手**　カテゴリの水準数が多いと、one-hot は列が爆発します。代わりに各水準を「その水準での**目的変数の平均**」で置きかえる手（ターゲットエンコーディング）もあります。強力ですが、目的変数を使うぶん**リークしやすい**のが弱点。必ず訓練データだけで平均を学ぶ・交差検証の内側で計算するといった用心が要ります。深入りはしませんが、「カテゴリの数値化は one-hot だけではない」と知っておくと引き出しが増えます。

In [ ]:
cat_cols = ['season', 'weathersit']
df[cat_cols].head(5)

### TODO③：カテゴリをワンホットに展開する

`df[cat_cols]` をワンホットエンコーディングして、season・weathersit の値ごとの 0/1 の列に展開してください。

pandas に、カテゴリ列を一発で展開してくれる関数があります。結果が True/False ではなく 0/1 の数値になるようにしてみましょう。

In [ ]:
# TODO: df[cat_cols] をワンホットエンコーディングして表示してください
# ヒント: pandas の関数を使います。カテゴリとして扱うため .astype('category')、0/1 にする引数（dtype=int）も使えます
...

## 4. 生の数値扱い vs ワンホットで回帰を比べる

カテゴリの扱い方だけを変えて、線形回帰のテストR²がどう変わるかを見ます。

### 深掘り：R²の差の読み方／係数の意味こそ本質／リークを防ぐ正しい順序

**R² の差をどう読むか**　下のセルでは、カテゴリの扱い方**だけ**を変えて線形回帰を比べます。結果は生の数値扱い **0.829** → ワンホット **0.844**。改善はささやか（+0.015 ほど）で、予測の当たり具合だけを見れば「誤差なら大差ない」と言えます。でも、この回の要点はそこではありません。

**係数の意味が正しくなる、が本質**　生の数値のままだと、たとえば `mnth`（月）を 1 増やすと台数が一定だけ増える、という**直線**を仮定してしまいます。ところが利用台数は冬に少なく夏・秋に多い**山なり**で、1 月から 12 月へまっすぐ増えたりはしません。ワンホットにすれば、各月・各季節が**それぞれ独立の効き目**を持てて、山なりの関係も素直に表せます。R² の小さな差の裏で、**係数の解釈が正しくなる**——これがワンホットの本当の収穫です。第1・2回で「数値の顔をしたカテゴリ」を生のまま入れて宙づりにしていた係数の、ここが後始末にあたります。

**つまずきどころ：リークを防ぐ正しい順序**　三点セット（補完・標準化・ワンホット）を通すときの順序が、この回いちばんの勘所です。正しくは、

1. **先に** 訓練・テストに**分割**する
2. 補完の中央値・標準化の $\mu,\sigma$・カテゴリの顔ぶれは、すべて**訓練データだけ**で学ぶ（`fit`）
3. テストには、その基準を**当てはめるだけ**（`transform`）

逆に、分割の前に整えてしまうと、テストの情報が代表値や平均・分散ににじみ込み、**リーク**します。2 節の検証で見たとおり、線形回帰では数値に出ないこともありますが、距離や正則化を使うモデルでは確かに効きます。だから**手法に依存せず「`fit` は訓練・`transform` はテスト」で統一**するのが安全です。

なぜ順序が大事かは、**前処理も「学習」の一部**だと見れば腑に落ちます。補完に使う中央値、標準化の $\mu,\sigma$、そして次に述べるカテゴリの顔ぶれ——どれも**データから計算して得た値**であって、モデルの係数と同じ「学習したパラメータ」です。だとすれば、これらをテストデータから計算する（`fit` する）のは、答えを盗み見してから試験を受けるのと同じ。前処理の道具にも「テストには `fit` しない」を徹底する、という一本の筋が通ります。

**カテゴリの顔ぶれも訓練から学んだ基準**　もしテストに、訓練で見なかった水準（たとえば豪雨のような未知の `weathersit`）が現れると、ワンホットで作られる**列の顔ぶれがズレて**モデルに渡せなくなります。「どんなカテゴリが存在するか」も、じつは訓練データから学んだ基準の一つ。逆に、テストにしか出ない水準は列として用意できないので、未知カテゴリをどう扱うか（無視するか、専用の「その他」列を設けるか）まで含めて設計が要ります。この列ぞろえの管理を、手作業のミス（列の付け忘れ・順序違い）ごと仕組みで安全に守る道具（ColumnTransformer やパイプライン）は、この演習シリーズの後の回で扱います。

**締め**　補完・標準化・ワンホットの三点セットを、**分割 → 訓練で学ぶ（`fit`）→ テストに当てる（`transform`）** の順で通せば、リークなしで評価できます。なお整えるのは説明変数だけで、目的変数（当てたい `cnt`）まで標準化しないよう注意——答えの単位を変えると解釈がややこしくなります。「**前処理も学習の一部**」——この一言が、この回のすべてを貫いています。

In [ ]:
y = df['cnt'].values
num = ['temp', 'atemp', 'hum', 'windspeed', 'yr', 'holiday', 'workingday']
cat = ['season', 'mnth', 'weekday', 'weathersit']
dfi = df.copy()
dfi[num] = SimpleImputer(strategy='median').fit_transform(dfi[num])

X_raw = dfi[num + cat].values  # カテゴリを生の数値のまま
X_oh = pd.concat([dfi[num],
    pd.get_dummies(dfi[cat].astype('category'), drop_first=True, dtype=float)], axis=1).values

for name, Xd in [('生の数値扱い', X_raw), ('ワンホット', X_oh)]:
    Xtr, Xte, ytr, yte = train_test_split(Xd, y, test_size=0.2, random_state=42)
    r2 = LinearRegression().fit(Xtr, ytr).score(Xte, yte)
    print(f'{name:12s} テストR²: {r2:.3f}')

## 目標に答えられたか

- 今日の目標は「欠損・単位ちがい・カテゴリ列を、解析できる形に整えたい」でした
- TODO①で、補完後のNaNは0になりましたか？ 削除（dropna）と比べて何日ぶん活かせたでしょうか？
- TODO②で、標準化後の平均・標準偏差はどうなりましたか？ 分布の「形」は変わったでしょうか？
- TODO③のワンホットと、生の数値扱いで、テストR²はどう違いましたか？ 数字の差は小さくても、なぜワンホットが正しいと言えるでしょうか？
- これらの前処理を、訓練とテストに分けたとき、どの順序で行うべきでしょうか？（→リーク防止）

## 今日の要点

- 欠測のある行をまるごと捨てると、その日の無傷だった列まで道連れになる。3列に1割の穴で 731 日中 197 日が消えた——だから削除より補完が基本
- 代表値で埋めるなら中央値が無難。順位の真ん中なので、裾を引く列でも数個の極端な値に吊り上げられない
- 埋めてよいかは「なぜ欠けたか」で決まる。MCAR なら中央値でよく、MAR なら条件つきで、MNAR では補完そのものが偏りを持ち込む
- 標準化は平行移動と拡大縮小だけなので、分布の形も順位も変えない。効くのは距離や正則化を使う手法で、しきい値で分ける木系には要らない
- 数値の顔をしたカテゴリはワンホットに開く。R² の差はわずかでも、ありもしない等間隔の順序を仮定せずに済み、係数が「基準カテゴリからの差」として読めるようになる
- 補完の中央値も、標準化の $\mu,\sigma$ も、カテゴリの顔ぶれも、データから計算した**学習したパラメータ**である。だから `fit` は訓練だけ、テストには `transform` だけ
- 線形回帰では標準化のリークが数値にまったく出なかった。**出ないことは、やってよいことを意味しない**——順序は手法によらず機械的に守る


## 次回へ

穴を埋め、土俵をそろえ、カテゴリを開く。第3回で組んだ「分けて測る」足場の上に、今日はその手前の工程を積みました。これで、現場から届いたままのデータをひととおり解析できる形に直し、しかもリークを踏まずに評価まで持っていけます。

ただし、今日ずっと物差しに使ってきたのは R² ひとつでした。生の数値扱い 0.829 とワンホット 0.844 のどちらがましかは言えても、「この予測は 1 日あたり何台ぶん外しているのか」「たまに大外しするのと、毎日少しずつ外すのと、どちらが困るのか」には何も答えていません。整え終わった次は、**測り方そのもの**を詰める段です。第5回では RMSE・MAE といった誤差の物差しを並べ、目的に合った一本を選べるようにします。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

4節のワンホット回帰では、`cat = ['season', 'mnth', 'weekday', 'weathersit']` の4列をワンホットにしました。
このうち `mnth`（月）を外して、同じ回帰をやり直してください。
ワンホット後の説明変数の**列数**と、**テストR²**の2つを表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

4節のコードをコピーして、`cat = [...]` の行だけ変えます。「生の数値扱い」の側は要らないので、ワンホットの側だけ残してよいです。
4節の `X_oh` を上書きしないよう、`X_oh2` のような別の名前にしておくと、あとで比べやすくなります。列数は `X_oh2.shape[1]` で分かります。

</details>


### 応用②（判断）

応用①で `mnth` を外したときの**テストR²**を、小数第3位まで（例: 0.812）で答えてください。
あわせて、4節のワンホット（0.844）からどれだけ変わったかも見ておきましょう。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

応用①で表示したテストR²を `round(r2, 3)` で小数第3位に丸めた値が答えです。
列数の方は、4節のワンホットが 29 列でしたから、`mnth` の分（11列）が減って 18 列になっているはずです。

</details>


### 応用③（解釈）

4節では `season`（季節）と `mnth`（月）を両方入れていました。応用①で `mnth` を外しても、テストR²はほとんど変わらなかったはずです。
季節は月でほぼ決まる（季節の切り替わる3・6・9・12月を除けば、月ごとに季節が一つに決まる）ことをふまえて、
「季節と月を両方入れる」ことにどんな問題があるかを、このデータ分析を頼んだ運営担当者に向けて3行で説明してください。
（この問題は第6回・第7回で「多重共線性」として正面から扱います。）


（ここに3行程度で書く）


## 発展（任意）

### KNNImputer：欠測を「似た行」から推定する

1節では、気温が欠けた日を**どの日も同じ中央値**で埋めました。手軽ですが、真夏の日も真冬の日も同じ気温になってしまいます。

ところが、`temp` が欠けた日でも、同じ日の `atemp`（体感気温）は残っています。体感気温が近い日を探して、その日の気温で埋めれば、ずっともっともらしい値になるはずです。

`KNNImputer` はこれを自動でやります。欠けていない列で行どうしの距離を測り、**近い k 行の平均**で穴を埋めます（k 近傍法の補完版です）。
距離を使うので、2節と同じくスケールに敏感です。標準化してから補完し、あとで単位を戻します。

1節で穴を空ける前の本当の値が分かっているので、中央値補完と KNN 補完の誤差を直接比べてみます。


In [ ]:
from sklearn.impute import KNNImputer

# 穴を空ける前の本当の値（1節と同じファイルを読み直す）
orig = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
mask = df[num_cols].isnull()          # 1節で空けた穴の位置

# 中央値補完
X_med = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(df[num_cols]), columns=num_cols)

# KNN 補完（標準化 → 補完 → 単位を戻す）。StandardScaler は NaN を無視して平均・標準偏差を学ぶ
sc = StandardScaler().fit(df[num_cols])
Z = KNNImputer(n_neighbors=5).fit_transform(sc.transform(df[num_cols]))
X_knn = pd.DataFrame(sc.inverse_transform(Z), columns=num_cols)

# 穴の位置だけで、本当の値との平均絶対誤差を比べる
print('列          中央値補完   KNN補完   (穴の数)')
for c in ['temp', 'hum', 'windspeed']:
    m = mask[c].values
    e_med = np.abs(X_med.loc[m, c] - orig.loc[m, c]).mean()
    e_knn = np.abs(X_knn.loc[m, c] - orig.loc[m, c]).mean()
    print(f'{c:10s}  {e_med:8.4f}   {e_knn:8.4f}   ({m.sum()})')


In [ ]:
# temp の穴について、本当の値と補完した値を散布図で比べる
m = mask['temp'].values
plt.scatter(orig.loc[m, 'temp'], X_med.loc[m, 'temp'], alpha=0.6, label='中央値補完')
plt.scatter(orig.loc[m, 'temp'], X_knn.loc[m, 'temp'], alpha=0.6, label='KNN補完')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='本当の値と一致する線')
plt.xlabel('本当の temp'); plt.ylabel('補完した temp'); plt.legend()
plt.show()


**読み方**

- `temp` の誤差は中央値補完の 0.1400 から KNN 補完の 0.0131 へ、**約10分の1**になりました。散布図でも、KNN の点は対角線にぴったり乗っています。同じ日の `atemp` がほぼ気温そのものなので、体感気温の近い日を探せば気温は当てられるのです。
- `hum` と `windspeed` の誤差は 0.1159 → 0.1125、0.0613 → 0.0568 と、ほとんど縮みません。湿度や風速を強く言い当てる列が、距離の計算に使った4列の中にないからです。
- つまり KNN 補完の良し悪しは「欠けた列と関係の深い列が、ほかに残っているか」で決まります。万能ではなく、中央値より遅く、標準化の手間も要ります。
- なお、ここでは全データで補完しましたが、本番では4節のとおり訓練データで `fit` してテストに `transform` する順序を守ります。

試すなら、`n_neighbors` を 3 や 20 に変えたり、距離に使う列に `season` や `mnth` を足したりして、誤差がどう動くか見てみましょう。
